# GitHub Repository Security Analysis

This notebook analyzes security vulnerabilities in popular GitHub repositories (2017-2025).

**Analysis includes:**
- CVE detection using Semgrep
- Insecure dependency scanning
- Vulnerability distribution graphs
- Trend analysis over time

## 1. Setup and Configuration

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone the repo so requirements.txt is available
    repo_url = "https://github.com/yourusername/AAR-Research-Paper-AI-Effects.git"
    repo_dir = "/content/AAR-Research-Paper-AI-Effects"

    if not os.path.exists(repo_dir):
        os.system(f"git clone {repo_url} {repo_dir}")

    os.chdir(repo_dir)

    # Install Semgrep system dependency
    os.system("apt-get install -q -y semgrep 2>/dev/null || pip install -q semgrep")

# Install Python requirements from requirements.txt
os.system(f"{sys.executable} -m pip install -q -r requirements.txt")

print("Environment ready.")

In [ ]:
import json
import subprocess
import tempfile
import shutil
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# Data processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# GitHub API
from github import RateLimitExceededException
from github.GithubException import GithubException

# Utilities
from tqdm.notebook import tqdm
import joblib
import time
import requests

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("All imports successful!")

In [ ]:
# Configuration
CONFIG = {
    'min_stars': 500,                    # Minimum stars for repository
    'min_contributors': 10,              # Minimum contributors
    'start_year': 2017,                  # Analysis start year
    'end_year': 2025,                    # Analysis end year
    'languages': ['Python', 'JavaScript', 'TypeScript', 'Java', 'Go', 'Ruby', 'PHP', 'C', 'C++'],
    'max_repos_per_language': 100,       # Max repos to fetch per language
    'clone_timeout': 300,                # Timeout for cloning (seconds)
    'semgrep_timeout': 600,              # Timeout for Semgrep scan (seconds)
    'data_dir': Path('./data'),          # Directory for cached data
    'repos_dir': Path('./repos'),        # Directory for cloned repos
    'results_dir': Path('./results'),    # Directory for results
}

# Create directories
for dir_path in [CONFIG['data_dir'], CONFIG['repos_dir'], CONFIG['results_dir']]:
    dir_path.mkdir(exist_ok=True)

print(f"Configuration loaded:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
from dotenv import load_dotenv

# Load token: keys.env locally, Colab Secrets when running in Colab
GITHUB_TOKEN = None

if IN_COLAB:
    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
        print("Loaded token from Colab Secrets.")
    except Exception:
        pass

if not GITHUB_TOKEN:
    load_dotenv('keys.env')
    GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')

if not GITHUB_TOKEN:
    raise ValueError(
        "GitHub token not found!\n"
        "  - Locally: copy keys.env.example to keys.env and add your token.\n"
        "  - Colab: add GITHUB_TOKEN to Secrets (key icon in left sidebar)."
    )

# Initialize GitHub client
from github import Github
github_client = Github(GITHUB_TOKEN, per_page=100)

rate_limit = github_client.get_rate_limit()
print(f"GitHub API Rate Limit:")
print(f"  Remaining: {rate_limit.core.remaining}/{rate_limit.core.limit}")
print(f"  Resets at: {rate_limit.core.reset}")

## 2. Repository Collection

In [ ]:
class RepositoryCollector:
    """Collects GitHub repositories based on specified criteria."""
    
    def __init__(self, github_client: Github, config: dict):
        self.client = github_client
        self.config = config
        self.cache_file = config['data_dir'] / 'repositories_cache.joblib'
    
    def search_repositories(self, language: str, year: int) -> List[dict]:
        """Search for repositories by language and creation year."""
        repos = []
        
        # Build search query
        query = (
            f"language:{language} "
            f"stars:>={self.config['min_stars']} "
            f"created:{year}-01-01..{year}-12-31"
        )
        
        try:
            results = self.client.search_repositories(
                query=query,
                sort='stars',
                order='desc'
            )
            
            count = 0
            for repo in results:
                if count >= self.config['max_repos_per_language']:
                    break
                
                # Check contributor count
                try:
                    contributors = list(repo.get_contributors()[:self.config['min_contributors'] + 1])
                    if len(contributors) < self.config['min_contributors']:
                        continue
                except GithubException:
                    continue
                
                repo_data = {
                    'full_name': repo.full_name,
                    'name': repo.name,
                    'owner': repo.owner.login,
                    'url': repo.html_url,
                    'clone_url': repo.clone_url,
                    'language': repo.language,
                    'stars': repo.stargazers_count,
                    'forks': repo.forks_count,
                    'contributors': len(contributors),
                    'created_at': repo.created_at.isoformat(),
                    'updated_at': repo.updated_at.isoformat(),
                    'description': repo.description,
                    'topics': repo.topics,
                    'year': year,
                }
                repos.append(repo_data)
                count += 1
                
        except RateLimitExceededException:
            print(f"Rate limit exceeded. Waiting...")
            self._wait_for_rate_limit()
            return self.search_repositories(language, year)
        except GithubException as e:
            print(f"GitHub API error: {e}")
        
        return repos
    
    def _wait_for_rate_limit(self):
        """Wait for rate limit to reset."""
        rate_limit = self.client.get_rate_limit()
        reset_time = rate_limit.core.reset
        wait_seconds = (reset_time - datetime.utcnow()).total_seconds() + 10
        if wait_seconds > 0:
            print(f"Waiting {wait_seconds:.0f} seconds for rate limit reset...")
            time.sleep(wait_seconds)
    
    def collect_all(self, use_cache: bool = True) -> pd.DataFrame:
        """Collect repositories for all languages and years."""
        
        # Check cache
        if use_cache and self.cache_file.exists():
            print("Loading repositories from cache...")
            return joblib.load(self.cache_file)
        
        all_repos = []
        years = range(self.config['start_year'], self.config['end_year'] + 1)
        languages = self.config['languages']
        
        total_iterations = len(languages) * len(years)
        
        with tqdm(total=total_iterations, desc="Collecting repositories") as pbar:
            for language in languages:
                for year in years:
                    pbar.set_postfix({'language': language, 'year': year})
                    repos = self.search_repositories(language, year)
                    all_repos.extend(repos)
                    pbar.update(1)
                    
                    # Save intermediate results
                    if len(all_repos) % 100 == 0:
                        df = pd.DataFrame(all_repos)
                        joblib.dump(df, self.cache_file)
        
        df = pd.DataFrame(all_repos)
        
        # Remove duplicates
        df = df.drop_duplicates(subset=['full_name'])
        
        # Save to cache
        joblib.dump(df, self.cache_file)
        
        print(f"\nCollected {len(df)} unique repositories")
        return df

In [ ]:
# Collect repositories
collector = RepositoryCollector(github_client, CONFIG)
repos_df = collector.collect_all(use_cache=True)

print(f"\nRepository Statistics:")
print(f"  Total repositories: {len(repos_df)}")
print(f"  Languages: {repos_df['language'].nunique()}")
print(f"  Years covered: {repos_df['year'].min()} - {repos_df['year'].max()}")
print(f"\nTop languages:")
print(repos_df['language'].value_counts().head(10))

## 3. Security Scanning with Semgrep

In [ ]:
class SemgrepScanner:
    """Scans repositories for CVE vulnerabilities using Semgrep."""
    
    def __init__(self, config: dict):
        self.config = config
        self.results_file = config['data_dir'] / 'semgrep_results.joblib'
        
        # Semgrep rulesets for security scanning
        self.rulesets = [
            'p/security-audit',
            'p/owasp-top-ten',
            'p/cwe-top-25',
            'p/python',
            'p/javascript',
            'p/typescript',
            'p/java',
            'p/go',
        ]
    
    def clone_repository(self, repo_info: dict) -> Optional[Path]:
        """Clone a repository to local directory."""
        repo_path = self.config['repos_dir'] / repo_info['full_name'].replace('/', '_')
        
        if repo_path.exists():
            return repo_path
        
        try:
            repo_path.parent.mkdir(parents=True, exist_ok=True)
            
            result = subprocess.run(
                ['git', 'clone', '--depth', '1', repo_info['clone_url'], str(repo_path)],
                capture_output=True,
                text=True,
                timeout=self.config['clone_timeout']
            )
            
            if result.returncode == 0:
                return repo_path
            else:
                print(f"Clone failed for {repo_info['full_name']}: {result.stderr}")
                return None
                
        except subprocess.TimeoutExpired:
            print(f"Clone timeout for {repo_info['full_name']}")
            if repo_path.exists():
                shutil.rmtree(repo_path)
            return None
        except Exception as e:
            print(f"Clone error for {repo_info['full_name']}: {e}")
            return None
    
    def scan_repository(self, repo_path: Path, repo_info: dict) -> dict:
        """Run Semgrep scan on a repository."""
        
        scan_result = {
            'full_name': repo_info['full_name'],
            'findings': [],
            'error': None,
            'scan_time': None,
        }
        
        start_time = time.time()
        
        try:
            # Run Semgrep with security rulesets
            cmd = [
                'semgrep',
                '--config', 'auto',
                '--json',
                '--timeout', '60',
                '--max-target-bytes', '1000000',
                str(repo_path)
            ]
            
            result = subprocess.run(
                cmd,
                capture_output=True,
                text=True,
                timeout=self.config['semgrep_timeout'],
                cwd=str(repo_path)
            )
            
            scan_result['scan_time'] = time.time() - start_time
            
            if result.stdout:
                try:
                    semgrep_output = json.loads(result.stdout)
                    findings = semgrep_output.get('results', [])
                    
                    for finding in findings:
                        scan_result['findings'].append({
                            'rule_id': finding.get('check_id', ''),
                            'severity': finding.get('extra', {}).get('severity', 'UNKNOWN'),
                            'message': finding.get('extra', {}).get('message', ''),
                            'file': finding.get('path', ''),
                            'line_start': finding.get('start', {}).get('line', 0),
                            'line_end': finding.get('end', {}).get('line', 0),
                            'cwe': finding.get('extra', {}).get('metadata', {}).get('cwe', []),
                            'owasp': finding.get('extra', {}).get('metadata', {}).get('owasp', []),
                        })
                except json.JSONDecodeError:
                    scan_result['error'] = 'Failed to parse Semgrep output'
            
        except subprocess.TimeoutExpired:
            scan_result['error'] = 'Semgrep scan timeout'
            scan_result['scan_time'] = self.config['semgrep_timeout']
        except Exception as e:
            scan_result['error'] = str(e)
            scan_result['scan_time'] = time.time() - start_time
        
        return scan_result
    
    def scan_all(self, repos_df: pd.DataFrame, use_cache: bool = True) -> List[dict]:
        """Scan all repositories."""
        
        # Check cache
        if use_cache and self.results_file.exists():
            print("Loading Semgrep results from cache...")
            return joblib.load(self.results_file)
        
        all_results = []
        
        with tqdm(total=len(repos_df), desc="Scanning repositories") as pbar:
            for _, repo_info in repos_df.iterrows():
                pbar.set_postfix({'repo': repo_info['name'][:20]})
                
                # Clone repository
                repo_path = self.clone_repository(repo_info.to_dict())
                
                if repo_path:
                    # Scan repository
                    result = self.scan_repository(repo_path, repo_info.to_dict())
                    all_results.append(result)
                    
                    # Clean up to save disk space (optional)
                    # shutil.rmtree(repo_path)
                
                pbar.update(1)
                
                # Save intermediate results
                if len(all_results) % 10 == 0:
                    joblib.dump(all_results, self.results_file)
        
        # Save final results
        joblib.dump(all_results, self.results_file)
        
        print(f"\nScanned {len(all_results)} repositories")
        return all_results

In [ ]:
# Run Semgrep scanning
scanner = SemgrepScanner(CONFIG)
semgrep_results = scanner.scan_all(repos_df, use_cache=True)

# Summary statistics
total_findings = sum(len(r['findings']) for r in semgrep_results)
repos_with_findings = sum(1 for r in semgrep_results if r['findings'])
errors = sum(1 for r in semgrep_results if r['error'])

print(f"\nSemgrep Scan Summary:")
print(f"  Total findings: {total_findings}")
print(f"  Repositories with findings: {repos_with_findings}")
print(f"  Scan errors: {errors}")

## 4. Dependency Vulnerability Scanning

In [ ]:
class DependencyScanner:
    """Scans repositories for insecure dependencies."""
    
    def __init__(self, config: dict):
        self.config = config
        self.results_file = config['data_dir'] / 'dependency_results.joblib'
    
    def find_dependency_files(self, repo_path: Path) -> dict:
        """Find dependency files in repository."""
        dep_files = {
            'python': [],
            'javascript': [],
            'ruby': [],
            'java': [],
            'go': [],
        }
        
        for path in repo_path.rglob('*'):
            if path.is_file():
                name = path.name.lower()
                
                # Python
                if name in ['requirements.txt', 'setup.py', 'pyproject.toml', 'pipfile.lock']:
                    dep_files['python'].append(path)
                
                # JavaScript/Node
                elif name in ['package.json', 'package-lock.json', 'yarn.lock']:
                    dep_files['javascript'].append(path)
                
                # Ruby
                elif name in ['gemfile', 'gemfile.lock']:
                    dep_files['ruby'].append(path)
                
                # Java
                elif name in ['pom.xml', 'build.gradle', 'build.gradle.kts']:
                    dep_files['java'].append(path)
                
                # Go
                elif name in ['go.mod', 'go.sum']:
                    dep_files['go'].append(path)
        
        return dep_files
    
    def scan_python_deps(self, requirements_file: Path) -> List[dict]:
        """Scan Python dependencies using pip-audit."""
        vulnerabilities = []
        
        try:
            result = subprocess.run(
                ['pip-audit', '-r', str(requirements_file), '--format', 'json'],
                capture_output=True,
                text=True,
                timeout=120
            )
            
            if result.stdout:
                try:
                    audit_results = json.loads(result.stdout)
                    for vuln in audit_results.get('dependencies', []):
                        if vuln.get('vulns'):
                            for v in vuln['vulns']:
                                vulnerabilities.append({
                                    'package': vuln.get('name', ''),
                                    'version': vuln.get('version', ''),
                                    'vulnerability_id': v.get('id', ''),
                                    'description': v.get('description', ''),
                                    'fix_versions': v.get('fix_versions', []),
                                    'ecosystem': 'python',
                                })
                except json.JSONDecodeError:
                    pass
        except Exception as e:
            pass
        
        return vulnerabilities
    
    def scan_npm_deps(self, package_json: Path) -> List[dict]:
        """Scan npm dependencies using npm audit."""
        vulnerabilities = []
        
        try:
            result = subprocess.run(
                ['npm', 'audit', '--json'],
                capture_output=True,
                text=True,
                timeout=120,
                cwd=package_json.parent
            )
            
            if result.stdout:
                try:
                    audit_results = json.loads(result.stdout)
                    for vuln_id, vuln in audit_results.get('vulnerabilities', {}).items():
                        vulnerabilities.append({
                            'package': vuln.get('name', vuln_id),
                            'version': vuln.get('range', ''),
                            'vulnerability_id': vuln_id,
                            'severity': vuln.get('severity', ''),
                            'description': vuln.get('title', ''),
                            'ecosystem': 'npm',
                        })
                except json.JSONDecodeError:
                    pass
        except Exception as e:
            pass
        
        return vulnerabilities
    
    def scan_repository(self, repo_path: Path, repo_info: dict) -> dict:
        """Scan all dependencies in a repository."""
        
        result = {
            'full_name': repo_info['full_name'],
            'vulnerabilities': [],
            'dependency_files_found': {},
        }
        
        if not repo_path or not repo_path.exists():
            return result
        
        dep_files = self.find_dependency_files(repo_path)
        result['dependency_files_found'] = {k: len(v) for k, v in dep_files.items()}
        
        # Scan Python dependencies
        for req_file in dep_files['python']:
            if req_file.name == 'requirements.txt':
                vulns = self.scan_python_deps(req_file)
                result['vulnerabilities'].extend(vulns)
        
        # Scan npm dependencies
        for pkg_file in dep_files['javascript']:
            if pkg_file.name == 'package.json':
                vulns = self.scan_npm_deps(pkg_file)
                result['vulnerabilities'].extend(vulns)
        
        return result
    
    def scan_all(self, repos_df: pd.DataFrame, use_cache: bool = True) -> List[dict]:
        """Scan dependencies for all repositories."""
        
        if use_cache and self.results_file.exists():
            print("Loading dependency results from cache...")
            return joblib.load(self.results_file)
        
        all_results = []
        
        with tqdm(total=len(repos_df), desc="Scanning dependencies") as pbar:
            for _, repo_info in repos_df.iterrows():
                pbar.set_postfix({'repo': repo_info['name'][:20]})
                
                repo_path = self.config['repos_dir'] / repo_info['full_name'].replace('/', '_')
                
                if repo_path.exists():
                    result = self.scan_repository(repo_path, repo_info.to_dict())
                    all_results.append(result)
                
                pbar.update(1)
                
                if len(all_results) % 10 == 0:
                    joblib.dump(all_results, self.results_file)
        
        joblib.dump(all_results, self.results_file)
        
        print(f"\nScanned dependencies for {len(all_results)} repositories")
        return all_results

In [ ]:
# Run dependency scanning
dep_scanner = DependencyScanner(CONFIG)
dependency_results = dep_scanner.scan_all(repos_df, use_cache=True)

# Summary
total_dep_vulns = sum(len(r['vulnerabilities']) for r in dependency_results)
repos_with_dep_vulns = sum(1 for r in dependency_results if r['vulnerabilities'])

print(f"\nDependency Scan Summary:")
print(f"  Total vulnerabilities: {total_dep_vulns}")
print(f"  Repositories with vulnerabilities: {repos_with_dep_vulns}")

## 5. Data Processing and Analysis

In [ ]:
def process_semgrep_results(semgrep_results: List[dict], repos_df: pd.DataFrame) -> pd.DataFrame:
    """Process Semgrep results into a DataFrame."""
    
    findings_data = []
    
    for result in semgrep_results:
        repo_name = result['full_name']
        
        # Get repo metadata
        repo_info = repos_df[repos_df['full_name'] == repo_name].iloc[0] if len(
            repos_df[repos_df['full_name'] == repo_name]) > 0 else None
        
        for finding in result['findings']:
            finding_data = {
                'repo_name': repo_name,
                'rule_id': finding['rule_id'],
                'severity': finding['severity'],
                'message': finding['message'],
                'file': finding['file'],
                'line_start': finding['line_start'],
                'cwe': ','.join(finding['cwe']) if finding['cwe'] else '',
                'owasp': ','.join(finding['owasp']) if finding['owasp'] else '',
            }
            
            if repo_info is not None:
                finding_data.update({
                    'language': repo_info['language'],
                    'stars': repo_info['stars'],
                    'year': repo_info['year'],
                    'contributors': repo_info['contributors'],
                })
            
            findings_data.append(finding_data)
    
    return pd.DataFrame(findings_data)


def process_dependency_results(dependency_results: List[dict], repos_df: pd.DataFrame) -> pd.DataFrame:
    """Process dependency scan results into a DataFrame."""
    
    vulns_data = []
    
    for result in dependency_results:
        repo_name = result['full_name']
        
        repo_info = repos_df[repos_df['full_name'] == repo_name].iloc[0] if len(
            repos_df[repos_df['full_name'] == repo_name]) > 0 else None
        
        for vuln in result['vulnerabilities']:
            vuln_data = {
                'repo_name': repo_name,
                'package': vuln['package'],
                'version': vuln['version'],
                'vulnerability_id': vuln['vulnerability_id'],
                'description': vuln.get('description', ''),
                'ecosystem': vuln['ecosystem'],
                'severity': vuln.get('severity', ''),
            }
            
            if repo_info is not None:
                vuln_data.update({
                    'language': repo_info['language'],
                    'stars': repo_info['stars'],
                    'year': repo_info['year'],
                })
            
            vulns_data.append(vuln_data)
    
    return pd.DataFrame(vulns_data)

In [ ]:
# Process results
findings_df = process_semgrep_results(semgrep_results, repos_df)
dep_vulns_df = process_dependency_results(dependency_results, repos_df)

print(f"Processed Findings:")
print(f"  Semgrep findings: {len(findings_df)}")
print(f"  Dependency vulnerabilities: {len(dep_vulns_df)}")

# Save processed data
findings_df.to_csv(CONFIG['results_dir'] / 'semgrep_findings.csv', index=False)
dep_vulns_df.to_csv(CONFIG['results_dir'] / 'dependency_vulnerabilities.csv', index=False)
repos_df.to_csv(CONFIG['results_dir'] / 'repositories.csv', index=False)

## 6. Visualization and Graphs

In [ ]:
# Set up figure directory
figures_dir = CONFIG['results_dir'] / 'figures'
figures_dir.mkdir(exist_ok=True)

### 6.1 Vulnerability Distribution by Severity

In [ ]:
if len(findings_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Severity distribution
    severity_counts = findings_df['severity'].value_counts()
    colors = {'ERROR': '#e74c3c', 'WARNING': '#f39c12', 'INFO': '#3498db'}
    bar_colors = [colors.get(s, '#95a5a6') for s in severity_counts.index]
    
    axes[0].bar(severity_counts.index, severity_counts.values, color=bar_colors)
    axes[0].set_xlabel('Severity')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Distribution of Findings by Severity')
    
    # Pie chart
    axes[1].pie(severity_counts.values, labels=severity_counts.index, autopct='%1.1f%%',
                colors=bar_colors)
    axes[1].set_title('Severity Distribution (%)')
    
    plt.tight_layout()
    plt.savefig(figures_dir / 'severity_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No findings data available for visualization.")

### 6.2 Vulnerabilities by Programming Language

In [ ]:
if len(findings_df) > 0 and 'language' in findings_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Findings by language
    lang_counts = findings_df['language'].value_counts().head(10)
    
    sns.barplot(x=lang_counts.values, y=lang_counts.index, ax=axes[0], palette='viridis')
    axes[0].set_xlabel('Number of Findings')
    axes[0].set_ylabel('Language')
    axes[0].set_title('Top 10 Languages by Vulnerability Count')
    
    # Findings per repository by language
    repos_per_lang = repos_df['language'].value_counts()
    findings_per_lang = findings_df['language'].value_counts()
    
    common_langs = set(repos_per_lang.index) & set(findings_per_lang.index)
    avg_findings = pd.Series(
        {lang: findings_per_lang.get(lang, 0) / repos_per_lang.get(lang, 1) 
         for lang in common_langs}
    ).sort_values(ascending=False).head(10)
    
    sns.barplot(x=avg_findings.values, y=avg_findings.index, ax=axes[1], palette='magma')
    axes[1].set_xlabel('Average Findings per Repository')
    axes[1].set_ylabel('Language')
    axes[1].set_title('Average Vulnerabilities per Repo by Language')
    
    plt.tight_layout()
    plt.savefig(figures_dir / 'language_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No language data available for visualization.")

### 6.3 Vulnerability Trends Over Time (2017-2025)

In [ ]:
if len(findings_df) > 0 and 'year' in findings_df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # Total findings by year
    yearly_findings = findings_df.groupby('year').size()
    axes[0, 0].plot(yearly_findings.index, yearly_findings.values, marker='o', linewidth=2)
    axes[0, 0].fill_between(yearly_findings.index, yearly_findings.values, alpha=0.3)
    axes[0, 0].set_xlabel('Year')
    axes[0, 0].set_ylabel('Total Findings')
    axes[0, 0].set_title('Total Vulnerabilities by Repository Creation Year')
    
    # Findings by severity over time
    severity_by_year = findings_df.groupby(['year', 'severity']).size().unstack(fill_value=0)
    severity_by_year.plot(kind='bar', stacked=True, ax=axes[0, 1], 
                          color=['#e74c3c', '#3498db', '#f39c12'])
    axes[0, 1].set_xlabel('Year')
    axes[0, 1].set_ylabel('Findings')
    axes[0, 1].set_title('Vulnerability Severity Distribution by Year')
    axes[0, 1].legend(title='Severity')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Average findings per repo by year
    repos_by_year = repos_df.groupby('year').size()
    avg_findings_year = yearly_findings / repos_by_year
    axes[1, 0].bar(avg_findings_year.index, avg_findings_year.values, color='steelblue')
    axes[1, 0].set_xlabel('Year')
    axes[1, 0].set_ylabel('Avg Findings per Repo')
    axes[1, 0].set_title('Average Vulnerabilities per Repository by Year')
    
    # Cumulative findings
    cumulative = yearly_findings.cumsum()
    axes[1, 1].plot(cumulative.index, cumulative.values, marker='s', linewidth=2, color='darkgreen')
    axes[1, 1].fill_between(cumulative.index, cumulative.values, alpha=0.3, color='green')
    axes[1, 1].set_xlabel('Year')
    axes[1, 1].set_ylabel('Cumulative Findings')
    axes[1, 1].set_title('Cumulative Vulnerability Growth Over Time')
    
    plt.tight_layout()
    plt.savefig(figures_dir / 'temporal_trends.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No temporal data available for visualization.")

### 6.4 Top CWE Categories

In [ ]:
if len(findings_df) > 0 and 'cwe' in findings_df.columns:
    # Extract and count CWE entries
    cwe_list = []
    for cwe_str in findings_df['cwe'].dropna():
        if cwe_str:
            cwes = cwe_str.split(',')
            cwe_list.extend([c.strip() for c in cwes if c.strip()])
    
    if cwe_list:
        cwe_counts = pd.Series(cwe_list).value_counts().head(15)
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        bars = ax.barh(range(len(cwe_counts)), cwe_counts.values, color=plt.cm.Reds(np.linspace(0.3, 0.9, len(cwe_counts))))
        ax.set_yticks(range(len(cwe_counts)))
        ax.set_yticklabels(cwe_counts.index)
        ax.invert_yaxis()
        ax.set_xlabel('Occurrences')
        ax.set_title('Top 15 CWE Categories Found')
        
        # Add value labels
        for bar, val in zip(bars, cwe_counts.values):
            ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, str(val), 
                   va='center', fontsize=10)
        
        plt.tight_layout()
        plt.savefig(figures_dir / 'cwe_distribution.png', dpi=300, bbox_inches='tight')
        plt.show()
    else:
        print("No CWE data available.")
else:
    print("No CWE data available for visualization.")

### 6.5 Dependency Vulnerability Analysis

In [ ]:
if len(dep_vulns_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # Vulnerabilities by ecosystem
    eco_counts = dep_vulns_df['ecosystem'].value_counts()
    axes[0, 0].pie(eco_counts.values, labels=eco_counts.index, autopct='%1.1f%%',
                   colors=plt.cm.Set2(np.linspace(0, 1, len(eco_counts))))
    axes[0, 0].set_title('Dependency Vulnerabilities by Ecosystem')
    
    # Top vulnerable packages
    pkg_counts = dep_vulns_df['package'].value_counts().head(15)
    sns.barplot(x=pkg_counts.values, y=pkg_counts.index, ax=axes[0, 1], palette='YlOrRd')
    axes[0, 1].set_xlabel('Vulnerability Count')
    axes[0, 1].set_ylabel('Package')
    axes[0, 1].set_title('Top 15 Most Vulnerable Packages')
    
    # Vulnerabilities by year (if available)
    if 'year' in dep_vulns_df.columns:
        yearly_dep_vulns = dep_vulns_df.groupby('year').size()
        axes[1, 0].bar(yearly_dep_vulns.index, yearly_dep_vulns.values, color='coral')
        axes[1, 0].set_xlabel('Year')
        axes[1, 0].set_ylabel('Vulnerabilities')
        axes[1, 0].set_title('Dependency Vulnerabilities by Repository Year')
    
    # Severity distribution (if available)
    if 'severity' in dep_vulns_df.columns and dep_vulns_df['severity'].notna().any():
        sev_counts = dep_vulns_df['severity'].value_counts()
        axes[1, 1].bar(sev_counts.index, sev_counts.values, 
                       color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'][:len(sev_counts)])
        axes[1, 1].set_xlabel('Severity')
        axes[1, 1].set_ylabel('Count')
        axes[1, 1].set_title('Dependency Vulnerability Severity')
    
    plt.tight_layout()
    plt.savefig(figures_dir / 'dependency_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No dependency vulnerability data available for visualization.")

### 6.6 Interactive Plotly Visualizations

In [ ]:
if len(findings_df) > 0 and 'year' in findings_df.columns and 'language' in findings_df.columns:
    # Interactive heatmap: Language vs Year
    pivot_data = findings_df.groupby(['year', 'language']).size().reset_index(name='count')
    pivot_table = pivot_data.pivot(index='language', columns='year', values='count').fillna(0)
    
    fig = px.imshow(
        pivot_table,
        labels=dict(x="Year", y="Language", color="Findings"),
        title="Vulnerability Heatmap: Language vs Year",
        color_continuous_scale="YlOrRd"
    )
    fig.update_layout(height=600)
    fig.write_html(figures_dir / 'heatmap_interactive.html')
    fig.show()

In [ ]:
if len(findings_df) > 0:
    # Sunburst chart: Language -> Severity -> Rule
    sunburst_data = findings_df.groupby(['language', 'severity', 'rule_id']).size().reset_index(name='count')
    sunburst_data = sunburst_data.head(100)  # Limit for readability
    
    fig = px.sunburst(
        sunburst_data,
        path=['language', 'severity', 'rule_id'],
        values='count',
        title='Vulnerability Hierarchy: Language -> Severity -> Rule',
        color='count',
        color_continuous_scale='RdYlGn_r'
    )
    fig.update_layout(height=700)
    fig.write_html(figures_dir / 'sunburst_interactive.html')
    fig.show()

In [ ]:
if len(repos_df) > 0 and len(findings_df) > 0:
    # Scatter plot: Stars vs Findings
    repo_findings = findings_df.groupby('repo_name').size().reset_index(name='finding_count')
    merged = repos_df.merge(repo_findings, left_on='full_name', right_on='repo_name', how='left')
    merged['finding_count'] = merged['finding_count'].fillna(0)
    
    fig = px.scatter(
        merged,
        x='stars',
        y='finding_count',
        color='language',
        size='contributors',
        hover_name='full_name',
        title='Repository Stars vs Vulnerability Count',
        labels={'stars': 'GitHub Stars', 'finding_count': 'Vulnerability Count'},
        log_x=True
    )
    fig.update_layout(height=600)
    fig.write_html(figures_dir / 'scatter_interactive.html')
    fig.show()

### 6.7 Summary Statistics

In [ ]:
# Generate summary report
summary = {
    'Total Repositories Analyzed': len(repos_df),
    'Total Semgrep Findings': len(findings_df),
    'Total Dependency Vulnerabilities': len(dep_vulns_df),
    'Repositories with Semgrep Findings': findings_df['repo_name'].nunique() if len(findings_df) > 0 else 0,
    'Repositories with Dep Vulnerabilities': dep_vulns_df['repo_name'].nunique() if len(dep_vulns_df) > 0 else 0,
    'Languages Analyzed': repos_df['language'].nunique(),
    'Years Covered': f"{repos_df['year'].min()} - {repos_df['year'].max()}",
}

if len(findings_df) > 0:
    summary['Most Common Severity'] = findings_df['severity'].mode().iloc[0] if len(findings_df['severity'].mode()) > 0 else 'N/A'
    summary['Most Affected Language'] = findings_df['language'].mode().iloc[0] if 'language' in findings_df.columns and len(findings_df['language'].mode()) > 0 else 'N/A'

summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
print("\n" + "="*50)
print("ANALYSIS SUMMARY")
print("="*50)
print(summary_df.to_string(index=False))

# Save summary
summary_df.to_csv(CONFIG['results_dir'] / 'summary.csv', index=False)

## 7. Export Results

In [ ]:
# Export all data
print("Exporting results...")

# Save as Excel with multiple sheets
with pd.ExcelWriter(CONFIG['results_dir'] / 'security_analysis_report.xlsx', engine='openpyxl') as writer:
    repos_df.to_excel(writer, sheet_name='Repositories', index=False)
    if len(findings_df) > 0:
        findings_df.to_excel(writer, sheet_name='Semgrep_Findings', index=False)
    if len(dep_vulns_df) > 0:
        dep_vulns_df.to_excel(writer, sheet_name='Dependency_Vulnerabilities', index=False)
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

print(f"\nResults exported to: {CONFIG['results_dir']}")
print(f"  - security_analysis_report.xlsx")
print(f"  - semgrep_findings.csv")
print(f"  - dependency_vulnerabilities.csv")
print(f"  - repositories.csv")
print(f"  - summary.csv")
print(f"\nFigures saved to: {figures_dir}")

## 8. Cleanup (Optional)

In [ ]:
# Uncomment to clean up cloned repositories
# import shutil
# if CONFIG['repos_dir'].exists():
#     shutil.rmtree(CONFIG['repos_dir'])
#     print("Cleaned up cloned repositories")